# Phase 2 — Step 5 v3: SMOTE + XGBoost Pathogenicity Classifier

**Inputs (from Google Drive):**
- `clinvar_training_data_v2.csv` — 80,692 ClinVar variants (training)
- `all_variants_enriched_step3.csv` — 1,031,175 CRISPR off-target variants (test)

**What this notebook does:**
1. Loads both files from Google Drive
2. Applies SMOTE to handle any class imbalance in training data
3. Trains XGBoost with 5-fold cross-validation
4. Plots ROC curve and feature importance
5. Predicts pathogenicity for all ~1M test variants (batched)
6. Compares RF (Step 4) vs XGBoost predictions
7. Saves full predictions + priority variants CSV

**Features used (6 — same as Step 4, no impact_enc):**
`cadd_phred`, `gerp_rs`, `sift_score`, `polyphen_score`, `spliceai_score`, `is_splice`

In [ ]:
!pip install xgboost imbalanced-learn scikit-learn pandas numpy matplotlib -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
os.makedirs('figures_step5', exist_ok=True)
print('✅ Libraries ready')

In [ ]:
from google.colab import drive
import pandas as pd, os

drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/My Drive/clinvar_training_data_v2.csv'
TEST_PATH  = '/content/drive/My Drive/all_variants_enriched_step3.csv'
RF_PATH    = '/content/drive/My Drive/ml_predictions_all_variants_rf.csv'

for path in [TRAIN_PATH, TEST_PATH]:
    if os.path.exists(path):
        print(f'✅ {os.path.basename(path)}  ({os.path.getsize(path)/1e6:.1f} MB)')
    else:
        print(f'❌ NOT FOUND: {path}')

print('\nLoading training data...')
train_df = pd.read_csv(TRAIN_PATH)
print(f'✅ Training : {len(train_df):,}  Labels: {train_df["label"].value_counts().to_dict()}')

print('Loading test data...')
var_df = pd.read_csv(TEST_PATH, low_memory=False)
print(f'✅ Test data: {len(var_df):,} variants')

# Load RF predictions if available for comparison
rf_available = os.path.exists(RF_PATH)
if rf_available:
    rf_df = pd.read_csv(RF_PATH, usecols=['chrom','pos','ref','alt',
                                           'pathogenicity_prob','risk_tier'])
    rf_df.columns = ['chrom','pos','ref','alt','prob_RF','risk_RF']
    print(f'✅ RF predictions loaded for comparison')
else:
    print('ℹ️  RF predictions not found in Drive — skipping comparison')

In [ ]:
import numpy as np, pandas as pd

# Single training feature — same as Step 4
FEATURE_COLS = ['cadd_phred']

def build_features(df):
    return df.copy()

train_df = build_features(train_df)
var_df   = build_features(var_df)

print(f'Single feature model: cadd_phred')
print(f'  Training shape: {train_df[FEATURE_COLS].shape}')
print(f'  Test shape    : {var_df[FEATURE_COLS].shape}')
print()
print('Training NaN:', train_df[FEATURE_COLS].isna().sum().to_dict())
print('Test NaN    :', var_df[FEATURE_COLS].isna().sum().to_dict())


In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

X_raw   = train_df[FEATURE_COLS].values
y_train = train_df['label'].values.astype(int)
X_var_r = var_df[FEATURE_COLS].values

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_raw)
X_var   = imputer.transform(X_var_r)

print(f'Training: {X_train.shape}  NaN: {np.isnan(X_train).sum()}')
print(f'Test    : {X_var.shape}   NaN: {np.isnan(X_var).sum()}')
print('✅ Imputation done')

In [ ]:
from imblearn.over_sampling import SMOTE

print(f'Before SMOTE: Pathogenic={y_train.sum():,}  Benign={(y_train==0).sum():,}')
smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)
X_sm, y_sm = smote.fit_resample(X_train, y_train)
print(f'After SMOTE : Pathogenic={y_sm.sum():,}  Benign={(y_sm==0).sum():,}')
print(f'Total SMOTE : {len(X_sm):,}')
print('✅ SMOTE done')

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

spw = (y_sm==0).sum() / max(y_sm.sum(), 1)
print(f'scale_pos_weight = {spw:.2f}')

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='logloss', random_state=42, n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_res = cross_validate(xgb, X_sm, y_sm, cv=cv,
    scoring=['accuracy','precision','recall','f1','roc_auc'])

print('\n5-FOLD CV RESULTS (XGBoost + SMOTE):')
print('='*45)
for m in ['accuracy','precision','recall','f1','roc_auc']:
    v = cv_res[f'test_{m}']
    print(f'  {m.upper():<12}: {v.mean():.4f} ± {v.std():.4f}')

print('\nFitting final model on SMOTE data...')
xgb.fit(X_sm, y_sm)
print('✅ XGBoost trained')

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_prob_cv = cross_val_predict(xgb, X_sm, y_sm, cv=cv, method='predict_proba')[:,1]
fpr, tpr, _ = roc_curve(y_sm, y_prob_cv)
auc_xgb = roc_auc_score(y_sm, y_prob_cv)

fig, ax = plt.subplots(figsize=(7,6))
ax.plot(fpr, tpr, color='#C0392B', lw=2.5, label=f'XGBoost+SMOTE (AUC={auc_xgb:.3f})')
ax.plot([0,1],[0,1], color='#AAAAAA', lw=1.5, ls='--', label='Random')
ax.fill_between(fpr, tpr, alpha=0.08, color='#C0392B')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve — XGBoost + SMOTE v3', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures_step5/roc_curve_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ AUC = {auc_xgb:.3f}')

In [ ]:
import matplotlib.pyplot as plt, numpy as np

feat_labels = ['CADD PHRED','GERP RS','SIFT Score',
               'PolyPhen2 Score','SpliceAI Score','Is Splice']
importances = xgb.feature_importances_
idx = np.argsort(importances)

colors = ['#C0392B' if importances[i]>0.10 else
          '#2E5FA3' if importances[i]>0.05 else '#7FB3D3' for i in idx]

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.barh([feat_labels[i] for i in idx], importances[idx],
               color=colors, height=0.6)
for bar, val in zip(bars, importances[idx]):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlabel('Feature Importance (XGBoost Gain)', fontsize=12)
ax.set_title('XGBoost + SMOTE Feature Importance v3', fontsize=12, fontweight='bold')
ax.set_xlim(0, importances.max()+0.08)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step5/feature_importance_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature importances:')
for i in np.argsort(importances)[::-1]:
    print(f'  {feat_labels[i]:<22}: {importances[i]:.3f}')

In [ ]:
import numpy as np

print(f'Predicting {len(var_df):,} variants in batches...')

BATCH = 100_000
proba_xgb = np.zeros(len(X_var))

for start in range(0, len(X_var), BATCH):
    end = min(start + BATCH, len(X_var))
    proba_xgb[start:end] = xgb.predict_proba(X_var[start:end])[:,1]
    print(f'  Predicted {end:>8,} / {len(X_var):,}', end='\r')

def risk_tier(p):
    return 'HIGH' if p >= 0.75 else 'MODERATE' if p >= 0.50 else 'LOW'

var_df['prob_XGB_SMOTE']   = np.round(proba_xgb, 4)
var_df['prediction_XGB']   = ['Pathogenic' if p>=0.5 else 'Benign' for p in proba_xgb]
var_df['risk_tier_XGB']    = [risk_tier(p) for p in proba_xgb]

print(f'\n\n✅ Predictions complete')
print()
print('XGBoost PREDICTION SUMMARY')
print('='*40)
tier_counts = var_df['risk_tier_XGB'].value_counts()
for tier in ['HIGH','MODERATE','LOW']:
    cnt = tier_counts.get(tier, 0)
    pct = cnt / len(var_df) * 100
    print(f'  {tier:<10}: {cnt:>8,}  ({pct:.2f}%)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('XGBoost + SMOTE Predictions — ~1M CRISPR Off-Target Variants v3',
             fontsize=13, fontweight='bold')

# Plot 1: probability histogram
ax = axes[0]
ax.hist(var_df['prob_XGB_SMOTE'], bins=50, color='#C0392B', edgecolor='white', alpha=0.85)
ax.axvline(0.75, color='#C0392B', ls='--', lw=1.5, label='HIGH threshold')
ax.axvline(0.50, color='#E67E22', ls='--', lw=1.5, label='MOD threshold')
ax.set_xlabel('Pathogenicity Probability')
ax.set_ylabel('Variant Count')
ax.set_title('Probability Distribution', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Plot 2: risk tier counts
ax = axes[1]
tier_order  = ['HIGH','MODERATE','LOW']
tier_colors = ['#C0392B','#E67E22','#27AE60']
tier_vals   = [var_df['risk_tier_XGB'].value_counts().get(t,0) for t in tier_order]
bars = ax.bar(tier_order, tier_vals, color=tier_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, tier_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(tier_vals)*0.01,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.set_title('Risk Tier Counts (XGBoost)', fontweight='bold')
ax.set_ylabel('Variant Count')
ax.spines[['top','right']].set_visible(False)

# Plot 3: RF vs XGB agreement on HIGH risk
ax = axes[2]
if rf_available:
    merged = var_df[['chrom','pos','ref','alt','risk_tier_XGB']].merge(
        rf_df, on=['chrom','pos','ref','alt'], how='inner')
    both_high  = ((merged['risk_tier_XGB']=='HIGH') & (merged['risk_RF']=='HIGH')).sum()
    xgb_only   = ((merged['risk_tier_XGB']=='HIGH') & (merged['risk_RF']!='HIGH')).sum()
    rf_only    = ((merged['risk_tier_XGB']!='HIGH') & (merged['risk_RF']=='HIGH')).sum()
    labels = ['Both HIGH\n(consensus)','XGB only\nHIGH','RF only\nHIGH']
    vals   = [both_high, xgb_only, rf_only]
    cols   = ['#C0392B','#E67E22','#2E5FA3']
    bars2  = ax.bar(labels, vals, color=cols, edgecolor='white', width=0.5)
    for bar, val in zip(bars2, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
                f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    ax.set_title('RF vs XGBoost HIGH Risk Agreement', fontweight='bold')
    ax.set_ylabel('Variant Count')
    ax.spines[['top','right']].set_visible(False)
else:
    ax.text(0.5, 0.5, 'RF predictions\nnot available', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title('RF vs XGB Agreement', fontweight='bold')

plt.tight_layout()
plt.savefig('figures_step5/prediction_distribution_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved')

In [ ]:
from google.colab import files
import os

save_cols = [
    'chrom','pos','ref','alt','gene','variant_type','impact_severity',
    'cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score',
    'is_splice','is_novel','in_controls','num_control_samples',
    'num_edited_samples','max_aaf_all','novelty_class',
    'prob_XGB_SMOTE','prediction_XGB','risk_tier_XGB'
]
save_cols = [c for c in save_cols if c in var_df.columns]

# Full predictions
out_full = 'ml_predictions_all_variants_xgb.csv'
var_df[save_cols].to_csv(out_full, index=False)
print(f'✅ Full: {out_full}  ({os.path.getsize(out_full)/1e6:.1f} MB)')

# Priority only (HIGH + MODERATE)
out_prio = 'ml_predictions_priority_variants_xgb.csv'
priority = var_df[var_df['risk_tier_XGB'].isin(['HIGH','MODERATE'])].sort_values(
    'prob_XGB_SMOTE', ascending=False)
priority[save_cols].to_csv(out_prio, index=False)
print(f'✅ Priority: {out_prio}  ({len(priority):,} variants)')

files.download(out_prio)
files.download(out_full)

for f in os.listdir('figures_step5'):
    files.download(f'figures_step5/{f}')
    print(f'📥 {f}')

print()
print('STEP 5 COMPLETION SUMMARY')
print('='*55)
print(f'Model     : XGBoost + SMOTE (n=300, depth=6, lr=0.05)')
print(f'Training  : {len(train_df):,} ClinVar variants')
print(f'Test      : {len(var_df):,} CRISPR off-target variants')
print(f'AUC       : {auc_xgb:.3f}')
tc = var_df['risk_tier_XGB'].value_counts()
print(f'HIGH      : {tc.get("HIGH",0):,}')
print(f'MODERATE  : {tc.get("MODERATE",0):,}')
print(f'LOW       : {tc.get("LOW",0):,}')
print()
print('Output files:')
print('  ml_predictions_all_variants_xgb.csv')
print('  ml_predictions_priority_variants_xgb.csv')
print('Next step: Step 6 — Grid Search Hyperparameter Tuning')